# Notebook 03 (Participant): Add a New Problem Scaffold

Complete the `TODO` sections to build a minimal EngiBench-compatible toy problem.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Annotated

import numpy as np
from gymnasium import spaces

from engibench.constraint import bounded
from engibench.core import ObjectiveDirection
from engibench.core import OptiStep
from engibench.core import Problem


In [ ]:
class ToyDensityProblem(Problem[np.ndarray]):
    """Minimal toy problem scaffold for workshop teaching."""

    version = 0
    objectives = (("toy_cost", ObjectiveDirection.MINIMIZE),)

    @dataclass
    class Conditions:
        target_density: Annotated[float, bounded(lower=0.0, upper=1.0)] = 0.5

    @dataclass
    class Config(Conditions):
        resolution: Annotated[int, bounded(lower=4, upper=128)] = 16
        max_iter: Annotated[int, bounded(lower=1, upper=200)] = 20

    dataset_id = "IDEALLab/beams_2d_50_100_v0"  # placeholder to satisfy scaffold
    container_id = None

    def __init__(self, seed: int = 0, **kwargs):
        super().__init__(seed=seed)
        self.config = self.Config(**kwargs)
        self.conditions = self.Conditions(target_density=self.config.target_density)
        self.design_space = spaces.Box(
            low=0.0, high=1.0, shape=(self.config.resolution, self.config.resolution), dtype=np.float32
        )

    def simulate(self, design: np.ndarray, config: dict | None = None) -> np.ndarray:
        # TODO 1: implement toy objective
        # Suggested terms:
        # - density mismatch to target_density
        # - smoothness penalty based on np.diff
        raise NotImplementedError('Implement simulate')

    def optimize(self, starting_point: np.ndarray, config: dict | None = None):
        # TODO 2: implement simple iterative optimizer
        # - update design toward target density
        # - append OptiStep each iteration
        raise NotImplementedError('Implement optimize')

    def render(self, design: np.ndarray, *, open_window: bool = False):
        import matplotlib.pyplot as plt

        fig, ax = plt.subplots(figsize=(4, 4))
        ax.imshow(design, cmap="viridis", vmin=0, vmax=1)
        ax.set_title("ToyDensityProblem design")
        ax.axis("off")
        if open_window:
            plt.show()
        return fig, ax

    def random_design(self):
        d = self.np_random.random(self.design_space.shape).astype(np.float32)
        return d, -1

In [ ]:
# Run this cell after finishing TODOs in ToyDensityProblem
problem = ToyDensityProblem(seed=42, resolution=16, target_density=0.4, max_iter=10)
start, _ = problem.random_design()

print('design space:', problem.design_space)
print('objectives:', problem.objectives)
print('conditions:', problem.conditions)

viol = problem.check_constraints(start, config={'target_density': 0.4, 'resolution': 16, 'max_iter': 10})
print('constraint violations:', len(viol))

obj0 = problem.simulate(start, config={'target_density': 0.4})
opt_design, history = problem.optimize(start, config={'target_density': 0.4})
objf = problem.simulate(opt_design, config={'target_density': 0.4})

print('initial objective:', float(obj0[0]))
print('final objective:', float(objf[0]))
print('optimization steps:', len(history))

problem.render(opt_design)

## Mapping to real EngiBench contributions

This toy scaffold demonstrates the interface shape only. For real contributions:

1. Create `engibench/problems/<new_problem>/v0.py`
2. Implement real `simulate` and (optionally) `optimize`
3. Provide `conditions`, `design_space`, and `dataset_id`
4. Add docs and tests

Reference docs: [Adding a new problem](https://github.com/IDEALLab/EngiBench/blob/main/docs/tutorials/new_problem.md)
